# पाठ ०२ - माइक्रोसफ्ट एजेन्ट फ्रेमवर्क अन्वेषण

**माइक्रोसफ्ट एजेन्ट फ्रेमवर्क (MAF)** एआई एजेन्टहरू निर्माण गर्न एक एकीकृत फ्रेमवर्क हो। यसले चार मूलभूत निर्माण ब्लकहरू सहित सफा, संयोज्य वास्तुकला प्रदान गर्छ:

- **क्लाइन्ट** – एआई मोडेल अन्तबिन्दुमा जडान गर्छ र सञ्चार सम्हाल्छ
- **एजेन्ट** – निर्देशनहरू र उपकरण परिभाषासहित क्लाइन्टलाई आवरण गर्छ
- **उपकरणहरू** – मोडेलले बोलाउन सक्ने कस्टम कार्यहरूसँग एजेन्ट क्षमताहरू विस्तार गर्छ
- **सत्र** – बहु-बारी अन्तरक्रियाहरूको लागि वार्तालाप इतिहास राख्छ

यस पाठमा, हामी यी अवधारणाहरू प्रयोग गरेर गन्तव्य उपलब्धता जाँच गर्ने **यात्रा बुकिङ एजेन्ट** निर्माण गर्नेछौं।


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## एजेन्ट फ्रेमवर्क आर्किटेक्चरलाई बुझ्नु

माइक्रोसफ्ट एजेन्ट फ्रेमवर्कले तहगत आर्किटेक्चर अनुसरण गर्छ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लाइन्ट** – `FoundryChatClient` ले Azure OpenAI डिप्लोयमेन्टमा जडान गर्छ। यसले प्रमाणीकरण, अनुरोध स्वरूपण, र प्रतिक्रिया पार्सिङको जिम्मा लिन्छ।
2. **एजेन्ट** – क्लाइन्टबाट `provider.create_agent()` प्रयोग गरेर सिर्जना गरिएको, एजेन्टले मोडेल पहुँचलाई निर्देशनहरू (सिस्टम प्रॉम्प्ट) र उपकरणहरूसँग संयोजन गर्छ।
3. **उपकरणहरू** – `@tool` ले सजाइएको Python फंक्शनहरू जुन एजेन्टले कार्यहरू सञ्चालन गर्न वा डेटा प्राप्त गर्न आह्वान गर्न सक्छ।
4. **सत्र** – एक `AgentSession` वस्तु (`agent.create_session()` मार्फत सिर्जना गरिएको) जसले संवाद इतिहास भण्डारण गर्छ, जसले एजेन्टलाई अघिल्लो सन्दर्भ सम्झने बहु-चरण संवाद सक्षम बनाउँछ।

हरेक तहलाई चरणबद्ध रुपमा बनाऔं।


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool सजावटकर्ता सहित उपकरणहरू थप्दै

उपकरणहरूले एजेन्टहरूलाई पाठ उत्पादन बाहेक कार्यहरू लिन अनुमति दिन्छ। `@tool` सजावटकर्ताले सामान्य Python कार्यलाई त्यस्तो केहीमा परिणत गर्छ जुन एजेन्टले कल गर्न सक्छ।

मुख्य बुँदाहरू:
- मोडलले प्रत्येक प्यारामिटरलाई बुझ्नेगरी `Annotated[type, "description"]` प्रयोग गर्नुहोस्।
- डकस्ट्रिङले मोडलले हेर्ने उपकरणको व्याख्या बनाउँछ।
- `approval_mode="never_require"` भन्ने अर्थ हो उपकरणले प्रयोगकर्ता पुष्टि बिना स्वतः चल्छ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## उपकरणहरूसहित एजेन्ट सिर्जना गर्दै

अब हामी क्लाइन्ट, निर्देशनहरू, र उपकरणहरूलाई एउटा एजेन्टमा समाहित गर्छौं। `निर्देशनहरू` प्रणाली प्राम्प्टको रूपमा काम गर्छन् — तिनीहरूले एजेन्टको व्यक्तित्व र व्यवहार निर्धारित गर्छन्।


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रहरूसँग बहु-चरण संवादहरू

एक `AgentSession` (जो `agent.create_session()` मार्फत सिर्जना गरिएको छ)ले संवादमा भएका सबै सन्देशहरूको ट्रयाक राख्छ। प्रत्येक `agent.run()` कलमा एउटै सत्र पास गरेर, एजेन्टसँग सम्पूर्ण संवाद इतिहास पहुँच हुन्छ र उ पुराना सन्देशहरूलाई पनि सन्दर्भ दिन सक्छ।

हामी `tools=[check_destination_availability]` पास गर्छौं ताकि एजेन्ट प्रत्येक चरणमा हाम्रो उपलब्धता परीक्षकलाई कल गर्न सकोस्।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework का चार स्तम्भहरूको अन्वेषण गर्नुभयो:

| अवधारणा | तपाईंले के सिक्नुभयो |
|---------|------------------|
| **Client** | `FoundryChatClient` प्रमाणपत्र आधारित प्रमाणीकरण सँग Azure OpenAI सँग जडान हुन्छ |
| **Agent** | `provider.create_agent()` मोडल जडानलाई निर्देशनहरू र नामसँग बन्डल गर्छ |
| **Tools** | `@tool` सजावटकर्ता एजेन्टलाई कल गर्न पाइथन कार्यहरू प्रदर्शन गर्छ |
| **Session** | `agent.create_session()` बहु चरणमा संवाद इतिहास कायम राख्छ |

यी बिल्डिङ ब्लकहरूले प्राकृतिक संवाद गर्न, बाह्य कार्यहरू कल गर्न, र सन्दर्भ कायम राख्न सक्ने एजेन्टहरू सिर्जना गर्न मिलेर काम गर्छन् — पछि आउने पाठहरूमा थप उन्नत एजेन्टिक ढाँचाहरूको आधारशिला।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
